# Curso de Python para Economistas
## Trabajo Practico 2

### Fecha de entrega:
Sábado 08/10 a las 23:59hs

### Modalidad de entrega y trabajo:
- Se debe entregar por mensaje privado de slack copiando a los 5 profesores y a los 3 miembros del equipo
- El archivo se debe llamar PythEcon2022_tp2_APELLIDO1_APELLIDO2_APELLIDO3.ipynb 
- Se debe trabajar en grupos de a 3 personas (si alguien no logra encontrar grupo escribanos para ponerlo en contacto con otros compañeros)

### COVID-19: Simulación de contagio

Las herramientas computacionales son una gran ayuda al momento de reproducir simulaciones y poder evaluar distintos escenarios. Los ultimos años hemos transitando un periodo de mucha incertidumbre debido a la pandemia COVID-19. Desde el 2020, entender cómo y cuánto se podría aplanar la curva de contagios y dilatarla en el tiempo, para no colapsar las camas de UCI, los respiradores y el personal de salud, era uno de los mayores interrogantes para gobernantes y directores de los centros de salud. En este contexto, poder contar con información actualizada y modelos que puedan inferir los contagios de los próximos días, fueron un recurso indispensable para poder prevenir y estar preparados.

En este ejercicio, tomaremos uno de los modelos epidemiológicos más simples, capaces de capturar muchas de las características típicas de estos brotes: [el SIR](https://es.wikipedia.org/wiki/Modelo_SIR#:~:text=El%20modelo%20SIR%20es%20uno,t%C3%ADpicas%20de%20los%20brotes%20epid%C3%A9micos.&text=El%20modelo%20relaciona%20las%20variaciones,y%20el%20per%C3%ADodo%20infeccioso%20promedio).  Lo utilizaremos para simular la cantidad de nuevos casos que surgirán en las próximas semanas, en cada una de las comunas de la Ciudad de Buenos Aires.

El nombre del modelo 'SIR' proviene de las iniciales de los tres grupos de individuos que se distinguen por su condición frente a la enfermedad. Estos son:

- Población susceptible (S), individuos sin inmunidad al agente infeccioso, y que por tanto pueden ser infectados si son expuestos al agente infeccioso. 
- Población infectada (I), individuos que están infectados en un momento dado y pueden transmitir la infección a individuos de la población susceptible con la que entran en contacto. Para nuestras simulaciones subdividiremos este grupo en dos: llamaremos 'I0' a los que están cursando la primera semana de la enfermedad y 'I1' a quienes están en su segunda semana de la enfermedad y próximos a recibir el alta.
- Población recuperada y fallecidos (R), individuos que son inmunes a la infección (o fallecidos), y consecuentemente no afectan a la transmisión cuando entran en contacto con otros individuos. (En este grupo entrarían los vacunados también, pero por simplicidad del modelo asumiremos que todavía no existe vacuna)

El modelo relaciona las variaciones de las tres poblaciones (Susceptible, Infectada y Recuperada) a través de la tasa de infección y el período infeccioso promedio. Es decir, que las personas pasan de susceptibles a infectadas acorde a una tasa de infección que depende del número promedio de contactos por persona por día y la probabilidad de transmisión de la enfermedad en un contacto entre un sujeto susceptible y un infeccioso. Además, las personas pasan de infectadas a recuperadas al cabo del periodo infeccioso promedio. En nuestras simulaciones tomaremos estos dos valores como dados. Usaremos una tasa de infección igual a 0.5 y un período infeccioso igual a dos semanas. Sin embargo, es importante resaltar que en ejercicios de simulación más extensos, estos dos parámetros es importante probarlos y adaptarlos en las simulaciones para que ajusten lo mejor posible a la realidad observada en los datos de las semanas pasadas y en los datos de otras ciudades que ya transitaron esta etapa.

Nota: Si bien el presente ejercicio esta enmarcado en un posible uso de Python en la vida real, nuestro objetivo es que practiquen los conceptos recientemente aprendidos. Es por ello que el modelo epidemiológico y las simulaciones estarán simplificadas lo suficiente para no perder de vista nuestro objetivo.


In [1]:
import sys
import pandas as pd
import requests
import csv
from datetime import timedelta
import random

# Si no se tiene instalado alguno de estos paquetes, borrar el '#' para correr la siguientes líneas reemplazando la última palabra (por ejemplo, "pandas") por el nombre del paquete que no tengan:
#conda install --yes --prefix {sys.prefix} pandas

# Si no funciona, probar con esta línea:
#!{sys.executable} -m pip install pandas

In [2]:
# Este bloque de código importa los datos. No hay que modificar nada en él.
# Se trabajará con datos reales de la ciudad de Buenos Aires, dercargados de: https://cdn.buenosaires.gob.ar/datosabiertos/datasets/salud/casos-covid-19/casos_covid19.csv
# Asegurarse de correr este jupyter notebook en la misma carpeta donde se tengan guardados los tres archivos .csv que están en el repositorio.

ARCHIVOS = ["infectados0.csv", "infectados1.csv", "recuperados.csv"]

def obtengo_infectados_por_comuna(archivos=ARCHIVOS):

    '''
    Esta función devuelve comuna, género y edad de las personas infectadas de COVID en una determinada fecha.

    Input:
        Tres archivos .CSV con personas infectadas en la última semana, hace una
        semana y hace más de dos semanas (a este grupo los llamamos recuperados).

    Output:
        Tupla con tres listas de personas en los tres estados: I0, I1, R.
        infectados0 (lista): contiene una tupla por cada persona infectada esta 
            semana. Cada tupla tiene comuna, género y edad de cada persona.
        infectados1 (lista): contiene una tupla por cada persona infectada hace una 
            semana. Cada tupla tiene comuna, género y edad de cada persona.
        recuperados (lista): contiene una tupla por cada persona infectada hace dos 
            semanas. Cada tupla tiene comuna, género y edad de cada persona.
    '''

    listas_personas = []

    for archivo in archivos:

        with open(archivo, "r") as f:
            filas = csv.reader(f)

            personas = []
            for fila in filas:
                personas.append((fila[0], fila[1], fila[2]))

        listas_personas.append(personas)

    infectados0, infectados1, recuperados = listas_personas

    return (infectados0, infectados1, recuperados)

#### Ejercicio 1: 
La función obtengo_infectados_por_comuna devuelve tres listas de personas, donde cada persona esta representada con una tupla conteniendo su comuna, su genero y su edad. La primer lista contiene las personas infectadas la ultima semana, la segunda lista contiene las personas infectadas en la semana anterior, y la ultima lista contiene las personas recuperadas. 

Guarden las listas que devuelve la función obtengo_infectados_por_comuna con los siguientes nombres e impriman las primeras 5 filas de la lista 'infectados0' para entender mejor su contenido:
- infectados0 
- infectados1
- recuperados

In [3]:
# ESCRIBIR EL CÓDIGO ACÁ

datos_covid = obtengo_infectados_por_comuna(archivos=ARCHIVOS)

infectados0 = datos_covid[0]
infectados1 = datos_covid[1]
recuperados = datos_covid[2]

print("INFECTADOS 0:", infectados0[0:5])

INFECTADOS 0: [('8.0', 'femenino', '86.0'), ('5.0', 'femenino', '87.0'), ('3.0', 'femenino', '79.0'), ('7.0', 'femenino', '85.0'), ('5.0', 'masculino', '55.0')]


#### Ejercicio 2:
Creen un diccionario que contenga el numero de la comuna (en formato string) como claves y la suma de personas en cada comuna como valor.

In [4]:
def suma_personas_por_comuna(lista_de_personas):

    '''
    Crea un diccionario con comunas como claves y el total de las personas en cada comuna como valor.

    Input:
        lista_de_personas (lista compuesta de tuplas): comuna, género, edad.

    Output:
        dic_por_comuna (dic): comunas como claves y suma de personas como valor.
    '''

    dic_por_comuna = {}

    for persona in lista_de_personas:

        comuna = persona[0] 
        comuna = str(int(float(comuna)))

        # Crear una condición que, si la comuna no está en el diccionario, la agregue como llave y como valor cuente 1 ciudadano.
        # Y, si la comuna ya está en el diccionario, simplemente, sume 1 ciudadano nuevo al valor.
        # ESCRIBIR EL CÓDIGO ACÁ

        key = comuna in dic_por_comuna.keys()

        if key == False:
            dic_por_comuna[comuna] = 1

        else:
            dic_por_comuna[comuna] = dic_por_comuna[comuna] + 1

    return dic_por_comuna

A continuación usaremos nuestra función suma_personas_por_comuna() sobre las tres listas de personas que tenemos

In [5]:
i0_por_comuna = suma_personas_por_comuna(infectados0)
i1_por_comuna = suma_personas_por_comuna(infectados1)
r_por_comuna = suma_personas_por_comuna(recuperados)

In [6]:
# Observar como quedó uno de los diccionarios
i0_por_comuna

{'8': 754,
 '5': 446,
 '3': 673,
 '7': 665,
 '14': 479,
 '9': 426,
 '12': 355,
 '1': 725,
 '2': 338,
 '6': 377,
 '10': 344,
 '15': 489,
 '4': 877,
 '11': 294,
 '13': 350}

#### Código para importar la población de CABA:
A continuación, importaremos el archivo con la población en cada comuna de la Ciudad de Buenos Aires y también lo guardaremos como un diccionario con las comunas como claves y el total de población como valores.

Nota: Modifiqué el archivo para reducir la población total por comuna a un valor ficticio más chico. Esto generará representación de comunas mas chicas en el ejercicio 5 y podrán encontrar y resolver errores más fácilmente.

In [7]:
# Para leer este archivo al inicio del jupyter notebook, hemos importado el paquete csv que nos va a devolver una lista por cada línea del archivo y un elemento en la lista por cada valor separado con coma en el .csv.
# Abrir el archivo que contiene la población total por comuna (estimación al 2020).

with open("pob_por_comuna.csv", "r") as f:

    filas = csv.reader(f)    
    pob_por_comuna = {}
    
    for fila in filas:
        pob_por_comuna[fila[0]] = int(fila[1])

    # Si este pop da error, comentarlo con el '#', ver el diccionario final y eliminar cualquier clave que no sea una comuna.
    #pob_por_comuna.pop("\ufeffcomuna")

pob_total = pob_por_comuna["pob_total"]
pob_por_comuna.pop("pob_total")
pob_por_comuna.pop("ï»¿comuna")

pob_por_comuna

{'1': 21910,
 '2': 16807,
 '3': 19957,
 '4': 23225,
 '5': 19049,
 '6': 18738,
 '7': 23475,
 '8': 19925,
 '9': 17218,
 '10': 17668,
 '11': 20202,
 '12': 21296,
 '13': 24618,
 '14': 24047,
 '15': 19429}

#### Ejercicio 3:
A continuación construiremos un diccionario llamado sir_por_comuna donde las claves serán las comunas y los valores serán otro diccionario con los detalles de cada comuna. En los diccionarios con detalle de cada comuna tendremos las siguientes claves 'S', 'I0', 'I1', 'R', 'N' y sus respectivos valores. 

'S': numero de personas susceptible (S = N-R-I0-I1), 

'I0': numero de personas infectadas esta semana, 

'I1': numero de personas infectadas la semana pasada,

'R': numero de personas recuperadas,

'N': total de personas en la comuna

In [8]:
# Construyo diccionario sir_por_comuna

sir_por_comuna = {}

for comuna in pob_por_comuna.keys():
    sir_por_comuna[comuna] = {'N':pob_por_comuna[comuna], 'R':r_por_comuna[comuna], 'I0':i0_por_comuna[comuna], 'I1':i1_por_comuna[comuna]}
    sir_por_comuna[comuna]["S"] = sir_por_comuna[comuna]["N"] - sir_por_comuna[comuna]["R"] - sir_por_comuna[comuna]["I0"] - sir_por_comuna[comuna]["I1"]

sir_por_comuna

{'1': {'N': 21910, 'R': 5135, 'I0': 725, 'I1': 686, 'S': 15364},
 '2': {'N': 16807, 'R': 1251, 'I0': 338, 'I1': 342, 'S': 14876},
 '3': {'N': 19957, 'R': 3302, 'I0': 673, 'I1': 754, 'S': 15228},
 '4': {'N': 23225, 'R': 5965, 'I0': 877, 'I1': 1019, 'S': 15364},
 '5': {'N': 19049, 'R': 1913, 'I0': 446, 'I1': 434, 'S': 16256},
 '6': {'N': 18738, 'R': 1094, 'I0': 377, 'I1': 343, 'S': 16924},
 '7': {'N': 23475, 'R': 5232, 'I0': 665, 'I1': 640, 'S': 16938},
 '8': {'N': 19925, 'R': 5787, 'I0': 754, 'I1': 916, 'S': 12468},
 '9': {'N': 17218, 'R': 1813, 'I0': 426, 'I1': 467, 'S': 14512},
 '10': {'N': 17668, 'R': 1363, 'I0': 344, 'I1': 346, 'S': 15615},
 '11': {'N': 20202, 'R': 1251, 'I0': 294, 'I1': 363, 'S': 18294},
 '12': {'N': 21296, 'R': 1034, 'I0': 355, 'I1': 338, 'S': 19569},
 '13': {'N': 24618, 'R': 1262, 'I0': 350, 'I1': 416, 'S': 22590},
 '14': {'N': 24047, 'R': 1617, 'I0': 479, 'I1': 452, 'S': 21499},
 '15': {'N': 19429, 'R': 1802, 'I0': 489, 'I1': 424, 'S': 16714}}

#### Ejercicio 4:
A continuación, usaremos el diccionario sir_por_comuna para crear la representación de cada comuna. Las comunas serán representadas por listas donde cada uno de los elementos será el estado de una persona en dicha comuna. Además, las posiciones adyacentes (una a la izquierda y una a la derecha) de cada elemento de la lista serán considerados los vecinos de cada persona y serán los únicos que pueden contagiar a esa persona (Por ej. si la comuna 1 fuera así comuna_1 = ['S', 'S', 'R', 'I0', 'I1', 'S'], entonces, el vecino en la posición 1 (es decir, comuna_1[1]) no puede ser contagiado porque sus vecinos no están infectados (son individuos 'S' y 'R'). En cambio, el vecino en la ultima posición (es decir, comuna_1[-1]) si puede ser contagiado porque su vecino está infectado, es un individuo 'I1')).

El largo de la lista será igual al total de ciudadanos en dicha comuna. Y la cantidad de 'S' en la lista será el numero de personas susceptibles en esa comuna, y la cantidad de 'I0' en la lista será el numero de personas infectadas esta semana, y así sucesivamente con las 4 letras ('S', I0', 'I1', 'R') que catalogan el estado de las personas de la comuna.

La posición inicial de las personas sería ideal que estuviera dada por la realidad. Pero como la información pública no tiene datos acerca de direcciones de estas peronas, en esta simulación iniciaremos la representación de la comuna asignandole posiciones aleatorias a cada persona. Es decir que una forma fácil de resolver este ejercicio sería creando una lista donde tengamos la cantidad necesaria de cada una de las iniciales(S', 'R', 'I0', 'I1') y luego intercambiar las letras de posición con el paquete random. En este caso usaremos: random.shuffle(comuna)

Por ultimo, guardaremos las listas que representan el estado de la comuna en un diccionario llamado comunas. Las llaves del diccionario seran nuevamente los numeros de las comunas (como strings) y el valor sera la lista. ej: comunas = {'1': ['S', 'S', 'R', 'I0', 'I1', 'S'], '2': ['I0', 'I1', 'R', 'S', 'S', 'S', 'S', 'I1', 'S'], ...}



In [9]:
# Se crea un diccionario vacío:
comunas = {}

# Se itera sobre las claves de sir_por_comuna, que son los números de las comunas:
for sir_com in sir_por_comuna.keys():

    # Se inicia una lista vacía para representar una comuna:
    comuna = []

    # Se itera sobre el diccionario interno de cada comuna. Este contiene las iniciales de los estados como clave y el total de personas como valor:
    for estado, n in sir_por_comuna[sir_com].items():

        if estado == "N":
            continue

        else:
            # Agregar a la lista 'comuna' tantas iniciales como personas en ese estado haya en esa comuna:
            # ESCRIBIR EL CÓDIGO ACÁ
            i = n
            while i > 0:
                comuna.append(estado)
                i -= 1

    # Se aleatoriza el orden de la lista comuna para que el estado inicial de la comuna tenga a las personas infectadas uniformemente distribuidas en la lista:
    random.shuffle(comuna)
    comunas[sir_com] = comuna

A continuación impriman una comuna para ver como les quedó la representación.

In [10]:
# Prueba

print(comunas['1'][:100])

['S', 'S', 'R', 'S', 'S', 'S', 'S', 'S', 'S', 'R', 'S', 'R', 'R', 'R', 'S', 'S', 'S', 'I0', 'R', 'S', 'S', 'S', 'R', 'I1', 'R', 'S', 'R', 'S', 'S', 'S', 'R', 'S', 'S', 'R', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'R', 'S', 'R', 'R', 'S', 'R', 'S', 'R', 'S', 'S', 'S', 'S', 'I1', 'S', 'S', 'R', 'S', 'S', 'S', 'S', 'S', 'S', 'R', 'S', 'S', 'S', 'R', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'R', 'S', 'S', 'S', 'R', 'R', 'S', 'S', 'S', 'S', 'R', 'R', 'I1', 'R', 'S', 'S', 'R', 'S', 'R', 'I1', 'S', 'S']


#### Ejercicio 5:

Construyan una función que cuente la cantidad personas en cierto estado en una comuna. Esta es una función auxiliar que ahora puede ayudarlos a verificar si el ejercicio 5 construyó bien las comunas. Pero además es una función que cumplirá un rol importante en la simulación final.

In [11]:
def contar_estado(comuna, estado="I0"):

    '''
    Esta función cuenta la cantidad de personas del estado especificado en una comuna.

    Inputs:
        comuna (lista): el estado de todas las personas en la comuna.
        estado (str): indica si se quiere contar los S (suceptibles), I0 
                    (infectados esta semana), I1 (infectados hace una semana),
                    o R (recuperados).
    Output:
        personas_del_estado (int): personas del estado seleccionado.
    '''

    # Se inicia el conteo de personas en 0:
    personas_del_estado = 0

    # Iterar sobre la lista comuna:
    # ESCRIBIR EL CÓDIGO ACÁ
    for com in comuna:
        # Condicionar en que sea una persona en el estado deseado:
        # ESCRIBIR EL CÓDIGO ACÁ
        if com == estado:
            # Si cumplió la condición, sumar una persona a personas_del_estado:
            # ESCRIBIR EL CÓDIGO ACÁ
            personas_del_estado += 1

    return personas_del_estado

Prueben si su función realiza la tarea que ustedes querían de la siguiente forma (pueden probar con otros estados y otras comunas también):

In [12]:
# Prueba

for sir_com in sir_por_comuna.keys():
    for est in sir_por_comuna[sir_com].keys():
        if est == "N":
            continue
        else:
            if (contar_estado(comunas[sir_com], estado=est) == sir_por_comuna[sir_com][est]) == False:
                print("No coincide el resultado de la función contar_estado con los datos del diccionario sir_por_comuna para la comuna", sir_com, "y para el estado", est)

#### Ejercicio 6:
Creen una funcion que se llame 'es_infectado'. Esta función determinará si una persona que tiene un vecino en estado 'I0' o 'I1' es infectado o no. Para ello, la función tendrá como argumentos la comuna analizada, la posición del inidividuo analizado y la tasa de contagio. 

Además, a cada individuo le asignaremos un valor random de inmunidad al momento de analizarlo (usen random.random() para crear un valor que representará que tan fuerte esta su sistema inmunológico). Este valor de inmunidad lo compararemos con la tasa de contagio. Si es menor que la tasa_de_contagio, resultará infectado, sino no.

Notas: 
- Recuerden que la primer posición de una lista es la 0. 
- En esta simulacion asumiremos que la persona en la primer posición  se podra contagiar de la persona en la posición 1 y de la persona en la ultima posición. A su vez, la persona en la ultima posición se podra contagiar de la persona en la anteúlitma posición y de la persona en la posición 0. Es decir que asumiremos que la comuna es como un anillo, donde la primera y la ultima persona de la lista se tocan.

In [13]:
def es_infectado(comuna, posicion, tasa_de_contagio):

    '''
    Determinar si una persona es infectada o no.

    Inputs:
        comuna (list): el estado de todas las personas de la comuna al comienzo de la semana.
        posicion (int): la posición del inidividuo analizado.
        tasa_de_contagio (float): la probabilidad de ser infectado dado que tu vecino está infectado.

    Output:
         True si la persona se infectó, False caso contrario.
    '''

    # Se define la posición del vecino izquierdo y del derecho:
    pos_izq = posicion - 1
    pos_der = posicion + 1

    # Se chequea si es el vecino en la última posición. Si lo es, se define su vecino de la derecha a la persona en la posición 0 (la primera de la lista):
    if posicion == len(comuna)-1:
        pos_der = 0

    # Se evalúa si alguno de sus dos vecinos está infectado (ya sea I0 o I1):
    if comuna[pos_izq] in ("I0", "I1") or comuna[pos_der] in ("I0", "I1"):
        # Se compara la inmunidad de la persona (creada con un número random) con la tasa de contagio de esta enfermedad:
        if random.random() < tasa_de_contagio:
            # El primer return será el valor para cuando esta condición se cumplió y el segundo return para cuando la condición no se cumplió:
            return True #ESCRIBIR EL CÓDIGO ACÁ

    return False #ESCRIBIR EL CÓDIGO ACÁ

In [14]:
# Prueba

tasas_de_contagio = [0.2, 0.4, 0.6, 0.8, 1.0]

for sir_com in sir_por_comuna.keys():
   for tasa in tasas_de_contagio:
      print("Comuna", sir_com, "- Tasa de contagio de", tasa, ":", es_infectado(comunas[sir_com], 1, tasa))

Comuna 1 - Tasa de contagio de 0.2 : False
Comuna 1 - Tasa de contagio de 0.4 : False
Comuna 1 - Tasa de contagio de 0.6 : False
Comuna 1 - Tasa de contagio de 0.8 : False
Comuna 1 - Tasa de contagio de 1.0 : False
Comuna 2 - Tasa de contagio de 0.2 : False
Comuna 2 - Tasa de contagio de 0.4 : False
Comuna 2 - Tasa de contagio de 0.6 : False
Comuna 2 - Tasa de contagio de 0.8 : False
Comuna 2 - Tasa de contagio de 1.0 : False
Comuna 3 - Tasa de contagio de 0.2 : False
Comuna 3 - Tasa de contagio de 0.4 : False
Comuna 3 - Tasa de contagio de 0.6 : False
Comuna 3 - Tasa de contagio de 0.8 : False
Comuna 3 - Tasa de contagio de 1.0 : False
Comuna 4 - Tasa de contagio de 0.2 : False
Comuna 4 - Tasa de contagio de 0.4 : False
Comuna 4 - Tasa de contagio de 0.6 : False
Comuna 4 - Tasa de contagio de 0.8 : False
Comuna 4 - Tasa de contagio de 1.0 : False
Comuna 5 - Tasa de contagio de 0.2 : False
Comuna 5 - Tasa de contagio de 0.4 : False
Comuna 5 - Tasa de contagio de 0.6 : False
Comuna 5 - 

#### Ejercicio 7:
A continuación, simulen el paso de una semana. Al cabo de una semana, las personas que tenían estado I0 pasaran a I1, los de estado I1 pasaran a estar recuperados y los de estado R continuaran como recuperados. Para quienes estaban en estado S, debemos aplicarles la función es_infectado para saber si se contagiará o no al comienzo de esta nueva semana.

Esta función se debe llamar simular_una_semana() y sus atributos serán: la comuna a simular (es decir la lista que contiene el estado de todos los ciudadanos de la comuna) y la tasa de contagio. La función debe devolver una nueva lista con el estado de todas las personas de la comuna al comienzo de la semana t+1.


In [15]:
def simular_una_semana(comuna_s1, tasa_de_contagio):

    '''
    Mover la simulacion un día.

    Inputs:
        comuna_s1 (list): el estado de todas las personas de la comuna al comienzo de la semana.
        tasa_de_contagio (float): la probabilidad de ser infectado dado que tu vecino está infectado.

    Output:
        comuna_s2 (list): el estado de todas las personas de la comuna al comienzo de la semana t+1.
    '''

    # Se crea vacía la comuna que representará el estado de la comuna después de simular una semana de transcurso de la epidemia:
    comuna_s2 = []
 
    # Se itera sobre la comuna inicial (i serán las posiciones y val las iniciales):
    for i, val in enumerate(comuna_s1):

        if val == "R":
            comuna_s2.append("R")

        # Usar elif para chequear en comuna_s1 cada uno de los estados que faltan y, luego, agregar a la comuna_s2 el estado al que pasa:
        elif val == "I0":
            #ESCRIBIR EL CÓDIGO ACÁ
            comuna_s2.append("I1")

        #ESCRIBIR EL CÓDIGO ACÁ
        elif val == "I1":
            #ESCRIBIR EL CÓDIGO ACÁ
            comuna_s2.append("R")

        else:
            # Se usa la función es_infectado para ver si una person S se contagia:
            if es_infectado(comuna_s1, i, tasa_de_contagio):
                # Acción a realizar si la función es_infectado devolvió True:
                comuna_s2.append("I0") 
            else:
                # Acción a realizar si la función es_infectado devolvió False:
                comuna_s2.append("S")

    return comuna_s2

In [16]:
# Prueba

print(contar_estado(comunas["1"], estado="I0"))
simulacion = simular_una_semana(comunas["1"], 0.5)
print(contar_estado(simulacion, estado="I0"))

725
926


#### Ejercicio 8:
Para finalizar creen una función para correr la simulación durante varias semanas. Su función se debe llamar correr_simulacion y debe tomar como argumentos el estado inicial de la comuna, la semilla, el número máximo de semanas para simular (max_num_semanas) y la tasa de contagio. Además, debe devolver una tupla con el estado final de la comuna y el número de semanas simuladas (s).

##### Notas:

- max_num_semanas siempre debe ser mayor a 0. Es decir que su función debe ejecutar como mínimo la simulación de una semana completa. 

- Antes de la simulación de la 1er semana se debe fijar la semilla para la función random.

- Su simulación debe comenzar la semana 0 y contar el número de semanas simuladas. Por ejemplo, si su simulación comienza la semana 0 y alcanza las condiciones de finalización después de simular una semana, debería devolver 1 como el número de semanas. Por otro lado, si su simulación comienza la semana 0 y se ejecuta para la semana 0 y la semana 1, debería devolver 2 como el número de semanas simuladas.

- Recuerden que hay dos condiciones para dejar de correr la simulación: 

    a) que hayan pasado la cantidad de semanas fijadas en max_num_semanas o 

    b) que nadie en la comuna está infectado después de simular una semana determinada. Deben usar la función contar_estado para verificar esta condición y debe verificar esta condición después de simular una semana (¡no antes!).


In [17]:
SEMILLA = 5000
TASA_CONTAGIO = 0.5

def correr_simulacion(comuna, max_num_semanas, semilla=SEMILLA, tasa_de_contagio=TASA_CONTAGIO):

    '''
    Correr la simulación para todas las semanas (con el límite de tiempo definido en max_num_semanas).

    Inputs:
        comuna (list): el estado de todas las personas de la comuna al comienzo de la semana.
        semilla (int): el número inicial para la función random que se usará para la simulación.
        max_num_semanas (int): la máxima cantidad de semanas a simular.
        tasa_de_contagio (float): la probabilidad de ser infectado dado que tu vecino está infectado.

    Output:
        tuple (s, comuna_s2), donde
            s (int): semanas de simulación.
            comuna_s2 (list): el estado de todas las personas de la comuna al comienzo de la semana s.
    '''

    # Se asegura de que el número de semanas a simular sea positivo:
    assert max_num_semanas > 0

    # Se fija la semilla:
    random.seed(semilla)

    # Realizar una primer simulación con la función simular_una_semana y guardar el resultado con el nombre comuna_s:
    # ESCRIBIR EL CÓDIGO ACÁ
    comuna_s = simular_una_semana(comuna, tasa_de_contagio)

    # Definir la variable s y darle valor 1 por la simulación que se acaba de hacer:
    # ESCRIBIR EL CÓDIGO ACÁ
    s = 1

    # Hacer un condicional con la función contar_estado para verificar si sigue habiendo personas infectadas en la comuna:
    # ESCRIBIR EL CÓDIGO ACÁ
    if contar_estado(comuna_s, "I0") == 0 and contar_estado(comuna_s, "I1") == 0:
        # Si se cumplió la condición, aquí terminará la simulación:
        return (s, comuna_s)

    # Hacer un loop sobre la cantidad máxima de iteraciones usando range:
    # ESCRIBIR EL CÓDIGO ACÁ
    for iter in range(1, max_num_semanas):
        # Realizar otra simulación con la función simular_una_semana y guardar el resultado actualizando (o sea pisando) comuna_s:
        # ESCRIBIR EL CÓDIGO ACÁ
        comuna_s = simular_una_semana(comuna_s, tasa_de_contagio)
        s += 1
        # Hacer un condicional con la función contar_estado para verificar si sigue habiendo personas infectadas en la comuna:
        # ESCRIBIR EL CÓDIGO ACÁ
        if contar_estado(comuna_s, "I0") == 0 and contar_estado(comuna_s, "I1") == 0:
            # Si se cumplió la condición, aquí terminará la simulación:
            break

    return (s, comuna_s)

Para probar si la simulacion les salió bien corranla para una comuna variando la tasa de contagio. Les dejo un ejemplo:

In [ ]:
# Prueba

tasas_de_contagio = [0.2, 0.4, 0.6, 0.8, 1.0]
semanas = 10

for tasa in tasas_de_contagio:
    sim = correr_simulacion(comunas["1"], 10, semilla=SEMILLA, tasa_de_contagio=tasa)
    print("\nComuna 1 - Máximo de", semanas, "semanas - Semilla", SEMILLA, "- Tasa de contagio de", tasa, ":\n", sim)

In [19]:
# Pequeña comuna ficticia por si sirve para probar que el código esté funcionando bien:

mini_comuna = ["S", "R", "S", "I0", "S", "I0", "R", "S", "I1", "S", "S", "I0"]
tasas_de_contagio = [0.2, 0.4, 0.6, 0.8, 1.0]
semanas = 10

for tasa in tasas_de_contagio:
    sim = correr_simulacion(mini_comuna, semanas, semilla=SEMILLA, tasa_de_contagio=tasa)
    print("Minicomuna - Máximo de", semanas, "semanas - Semilla", SEMILLA, "- Tasa de contagio de", tasa, ":", sim)

Minicomuna - Máximo de 10 semanas - Semilla 5000 - Tasa de contagio de 0.2 : (4, ['S', 'R', 'S', 'R', 'S', 'R', 'R', 'S', 'R', 'S', 'R', 'R'])
Minicomuna - Máximo de 10 semanas - Semilla 5000 - Tasa de contagio de 0.4 : (4, ['R', 'R', 'R', 'R', 'S', 'R', 'R', 'S', 'R', 'S', 'S', 'R'])
Minicomuna - Máximo de 10 semanas - Semilla 5000 - Tasa de contagio de 0.6 : (4, ['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R'])
Minicomuna - Máximo de 10 semanas - Semilla 5000 - Tasa de contagio de 0.8 : (4, ['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R'])
Minicomuna - Máximo de 10 semanas - Semilla 5000 - Tasa de contagio de 1.0 : (3, ['R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R', 'R'])
